In [1]:
%pip install pandas numpy pyspark

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

Note: you may need to restart the kernel to use updated packages.


In [2]:
titanic_data = {
    'Survived': [0, 1, 1, 0, 1],
    'Pclass': [3, 1, 3, 1, 3],
    'Sex': ['male', 'female', 'female', 'male', 'female'],
    'Age': [22.0, 38.0, 26.0, 35.0, np.nan], # np.nan represents a missing age
    'Fare': [7.25, 71.28, 7.92, 53.1, 8.05]
}

# Create the DataFrame from the dictionary
titanic_df = pd.DataFrame(titanic_data)
mean_age = titanic_df["Age"].mean()
titanic_df["Age"] = titanic_df["Age"].fillna(mean_age)

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("spark://localhost:7077").appName("DataScienceProject").config("spark.sql.catalogImplementation", "in-memory").getOrCreate()

print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/07 11:25:15 WARN Utils: Your hostname, soc-5CG2243S91, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/07 11:25:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/07 11:25:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
spark_df = spark.createDataFrame(titanic_df)

In [11]:
spark_df.rdd.getNumPartitions()

8

In [6]:
spark_df.rdd.glom().collect()

[[],
 [Row(Survived=0, Pclass=3, Sex='male', Age=22.0, Fare=7.25)],
 [],
 [Row(Survived=1, Pclass=1, Sex='female', Age=38.0, Fare=71.28)],
 [Row(Survived=1, Pclass=3, Sex='female', Age=26.0, Fare=7.92)],
 [],
 [Row(Survived=0, Pclass=1, Sex='male', Age=35.0, Fare=53.1)],
 [Row(Survived=1, Pclass=3, Sex='female', Age=30.25, Fare=8.05)]]

In [12]:
spark_df.printSchema()

root
 |-- Survived: long (nullable = true)
 |-- Pclass: long (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Fare: double (nullable = true)



In [13]:
selected_df = spark_df.filter(spark_df["Age"] > 20)

In [14]:
spark_df.show()

+--------+------+------+-----+-----+
|Survived|Pclass|   Sex|  Age| Fare|
+--------+------+------+-----+-----+
|       0|     3|  male| 22.0| 7.25|
|       1|     1|female| 38.0|71.28|
|       1|     3|female| 26.0| 7.92|
|       0|     1|  male| 35.0| 53.1|
|       1|     3|female|30.25| 8.05|
+--------+------+------+-----+-----+



In [15]:
spark_df.distinct().show()

+--------+------+------+-----+-----+
|Survived|Pclass|   Sex|  Age| Fare|
+--------+------+------+-----+-----+
|       1|     3|female| 26.0| 7.92|
|       0|     1|  male| 35.0| 53.1|
|       0|     3|  male| 22.0| 7.25|
|       1|     1|female| 38.0|71.28|
|       1|     3|female|30.25| 8.05|
+--------+------+------+-----+-----+



In [19]:
spark_df.filter((spark_df["Survived"] == 1) & (spark_df["Sex"] == "female")).select("Age", "Fare").sort("Age", ascending=False).show()

+-----+-----+
|  Age| Fare|
+-----+-----+
| 38.0|71.28|
|30.25| 8.05|
| 26.0| 7.92|
+-----+-----+



In [ ]:
%pip install pyarrow 

Note: you may need to restart the kernel to use updated packages.


In [34]:
from pyspark.sql import functions as sf

spark_df.groupBy("Pclass").agg(sf.avg("Age").alias("mean_age")).withColumn("mean_age", sf.round("mean_age", 2)).show()

+------+--------+
|Pclass|mean_age|
+------+--------+
|     3|   26.08|
|     1|    36.5|
+------+--------+

